In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv("C:\\Users\\Narendra Kumar\\OneDrive\\Pictures\\Screenshots\\bank+marketing\\bank-full.csv")

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [8]:
# Customers kahan drop-off ho rahe hain?
df["converted"] = (df["y"] == "yes").astype(int)

contacted = len(df)
engaged   = (df["duration"] > 60).sum()
converted = df["converted"].sum()

print(f"Contacted: {contacted}")
print(f"Engaged: {engaged} ({engaged/contacted*100:.1f}%)")
print(f"Converted: {converted} ({converted/contacted*100:.1f}%)")

# Result: 89.5% engage ho jaate hain, lekin sirf 11.7% convert karte hain./
#  Matlab sabse bada drop-off Engaged ke baad hai, call attend hone ke baad bhi log deposit nahi le rahe.

# Action: Call ke opening 60 second theek hain, lekin baad mein pitch/offer weak hai. Sales script sudharo, jaldi value batao.

Contacted: 45211
Engaged: 40445 (89.5%)
Converted: 5289 (11.7%)


In [9]:
# Kaunsa channel achha hai?
df.groupby("contact")["converted"].mean() * 100

# Result: cellular 14.9%, telephone kam, unknown sirf 4.1%.

# Action: Zyada calls cellular se karo, aur "unknown" wali calls ka channel record karna shuru karo taaki future mein tracking better ho.

contact
cellular     14.918900
telephone    13.420509
unknown       4.070661
Name: converted, dtype: float64

In [10]:
# Kaunsa month/segment best hai?
df.groupby("month", observed=True)["converted"].mean().sort_values(ascending=False) * 100
df.groupby("job")["converted"].mean().sort_values(ascending=False) * 100

# Result: March, Sep, Oct, Dec mein conversion sabse zyada (40-52%), lekin May mein sirf 6.7% (jabki 30% saari calls yahi hoti hain). 
# Job mein student aur retired sabse zyada convert karte hain.

# Action: May mein calls kam karo, Sep-Dec mein badhao. Student aur retired logon ko priority do.

job
student          28.678038
retired          22.791519
unemployed       15.502686
management       13.755551
admin.           12.202669
self-employed    11.842939
unknown          11.805556
technician       11.056996
services          8.883004
housemaid         8.790323
entrepreneur      8.271688
blue-collar       7.274969
Name: converted, dtype: float64

In [11]:
# Baar-baar call karna kaam karta hai kya?

bins = pd.cut(df["campaign"], [0,1,2,3,5,10,100])
df.groupby(bins, observed=True)["converted"].mean() * 100



campaign
(0, 1]       14.597583
(1, 2]       11.203519
(2, 3]       11.193624
(3, 5]        8.626561
(5, 10]       6.521051
(10, 100]     3.929766
Name: converted, dtype: float64

In [ ]:
# Jo customers pehle kisi campaign mein "yes" bol chuke hain, unka is baar conversion kaisa hai?
df.groupby("poutcome")["converted"].mean().sort_values(ascending=False) * 100

# Result: success wale 64.7% convert karte hain — baaki sab se kahin zyada. 

# Action: Naye leads dhoondhne se pehle, purane successful customers ko phir se call karo.

poutcome
success    64.725347
other      16.684783
failure    12.609671
unknown     9.161503
Name: converted, dtype: float64

In [ ]:
#  Loan/balance ka asar
df.groupby("housing")["converted"].mean() * 100
df.groupby("loan")["converted"].mean() * 100

# Result: Jinke paas housing loan nahi hai unka conversion (16.7%) un logon se dugna hai jinke paas hai (7.7%).
#  Matlab loan-wale customers ko target karna kam faayde ka hai.

In [13]:
# Age group
bins = pd.cut(df["age"], [17,25,35,45,55,65,120])
df.groupby(bins, observed=True)["converted"].mean() * 100

age
(17, 25]     23.952096
(25, 35]     12.003083
(35, 45]      9.389434
(45, 55]      9.352744
(55, 65]     14.123885
(65, 120]    42.609854
Name: converted, dtype: float64

In [14]:
# Quality of leads ka proper comparision
g = df.groupby("contact").agg(calls=("converted","size"), customers=("converted","sum"))
g["conv_rate"] = g["customers"]/g["calls"]*100
g["share_of_effort"] = g["calls"]/len(df)*100